In [1]:
import os
import fitz  # PyMuPDF
from PIL import Image
import io
import base64
from openai import OpenAI

# ==========================================
# 1. 配置官方環境（對齊 README 照片）
# ==========================================
apertus_key = "sk-rc-qLyAj14iCtkrZu9FZuwgYA"
client = OpenAI(
    api_key=apertus_key,
    base_url="https://api.swissai.svc.cscs.ch/v1"  # 對齊你照片右下角的官方網址
)

current_dir = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy"
pdf_search_dir = "/home/renku/work/Durham-Hackathon-2026-w1t2"

# 建立兩個全新、乾淨的資料夾，把過去混亂的檔案隔開
clean_image_dir = os.path.join(current_dir, "extracted_pure_charts")
clean_caption_dir = os.path.join(current_dir, "image_captions_kimi_final")

os.makedirs(clean_image_dir, exist_ok=True)
os.makedirs(clean_caption_dir, exist_ok=True)

# 協助將圖片轉為 Base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# ==========================================
# 2. 步驟一：從全專案空間搜尋 4 個核心 PDF
# ==========================================
print("🕵️‍♂️ 正在搜尋工作區中的原始 PDF 檔案...")
pdf_list = []
for root, dirs, files in os.walk(pdf_search_dir):
    if ".ipynb_checkpoints" in root or "extracted" in root or "captions" in root:
        continue
    for file in files:
        if file.lower().endswith('.pdf'):
            pdf_list.append((file, os.path.join(root, file)))

print(f"🎯 成功鎖定 {len(pdf_list)} 個原始 PDF 報告。")

# ==========================================
# 3. 步驟二：精準提取圖表（結合物件抽圖與智慧頁面渲染）
# ==========================================
print("\n🚀 啟動『混合式圖表提取引擎』...")
total_images_saved = 0

for file_name, pdf_path in pdf_list:
    clean_pdf_name = file_name.replace(".pdf", "").replace(" ", "_")
    print(f"📂 正在深度解析報告: {file_name}")
    
    try:
        doc = fitz.open(pdf_path)
        
        # 針對特殊報告（如大名鼎鼎的 Swiss Re 向量圖報告）做全頁高畫質渲染
        # 針對一般報告則採用精準物件抽圖，這樣可以拿到最乾淨的直條圖、折線圖
        is_vector_report = "swissre" in file_name.lower() or "catastrophe" in file_name.lower()
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            
            if is_vector_report:
                # 向量報告：直接全頁高解析度拍照片，防止漏掉直條圖
                zoom = 2.0
                mat = fitz.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat, alpha=False)
                image = Image.open(io.BytesIO(pix.tobytes("png")))
                
                out_name = f"{clean_pdf_name}_page_{page_num + 1}_vector.png"
                image.save(os.path.join(clean_image_dir, out_name))
                total_images_saved += 1
            else:
                # 一般報告：只抽內嵌的真正圖片物件，避開純文字塊
                image_list = page.get_images(full=True)
                for img_index, img in enumerate(image_list):
                    xref = img[0]
                    base_image = doc.extract_image(xref)
                    try:
                        image = Image.open(io.BytesIO(base_image["image"]))
                        out_name = f"{clean_pdf_name}_page_{page_num + 1}_img_{img_index + 1}.{base_image['ext']}"
                        image.save(os.path.join(clean_image_dir, out_name))
                        total_images_saved += 1
                    except:
                        continue
                        
    except Exception as pdf_err:
        print(f"   ❌ 讀取此 PDF 失敗: {pdf_err}")

print(f"\n✨ 第一階段完工！全組共精確捕獲 {total_images_saved} 張圖表候選圖片。")
print(f"📍 實體圖片已擺放在: {clean_image_dir}")

# ==========================================
# 4. 步驟三：呼叫 Kimi-K2.5-SDSC 進行智能過濾與逆向描述
# ==========================================
print("\n🔮 進入第二階段：啟動官方指定頂級 Kimi 視覺大模型進行數據清洗...")

all_extracted_images = [f for f in os.listdir(clean_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
valid_chart_count = 0

for idx, img_file in enumerate(all_extracted_images):
    image_path = os.path.join(clean_image_dir, img_file)
    txt_output_path = os.path.join(clean_caption_dir, f"{os.path.splitext(img_file)[0]}_caption.txt")
    
    try:
        base64_image = encode_image(image_path)
        
        # 嚴格的過濾機制 Prompt
        filter_prompt = """
        You are an expert data analyst and an advanced data filtering assistant for a RAG system.
        Look at the provided image carefully.
        
        CRITICAL FILTER RULE:
        - If the image contains a visual data chart, diagram, or graph (such as a bar chart, line chart, trend graph, pie chart, scatter plot, timeline diagram, or data-heavy infographics), proceed to analyze it.
        - If the image is JUST a paragraph of plain text, a standard document page layout, a corporate logo, or a basic formatting decoration, respond with EXACTLY the word "[SKIP]" and nothing else.
        
        If it IS a valid chart/graph, reverse-engineer it completely:
        1. Chart Title / Theme: What data is being visualized?
        2. X-axis and Y-axis: Precise labels, categories, units, and ranges.
        3. Key Data Points & Trends: Extract exact numbers, percentage changes, turning points, and trends over time or categories.
        4. Main Takeaway: What is the primary analytical conclusion shown here?
        Be extremely precise, do not assume or hallucinate. Return your response in clean, organized markdown text.
        """
        
        # 呼叫你手指點著的最強 Kimi 視覺模型！
        response = client.chat.completions.create(
            model="moonshotai/Kimi-K2.5-SDSC",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": filter_prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                    ]
                }
            ],
            max_tokens=600
        )
        
        result_text = response.choices[0].message.content.strip()
        
        if "[SKIP]" in result_text or result_text == "SKIP":
            print(f"⏩ [{idx+1}/{len(all_extracted_images)}] {img_file} -> 經 Kimi 鑑定為非圖表物件，已智能過濾跳過。")
        else:
            valid_chart_count += 1
            with open(txt_output_path, "w", encoding="utf-8") as out_f:
                out_f.write(result_text)
            print(f"📈 [{idx+1}/{len(all_extracted_images)}] {img_file} -> ✅ 成功！Kimi 鑑定為【核心數據圖表】，語義描述已寫入！")
            
    except Exception as api_err:
        print(f"❌ 處理 {img_file} 時發生 API 連線異常: {api_err}")

print(f"\n🎉 狂賀！全新 Pipeline 完美收工！")
print(f"📊 最終幫你們組保留了 {valid_chart_count} 個真正的趨勢/直條圖核心數據說明書。")
print(f"📂 真正乾淨且精準的圖表描述文字都在：{clean_caption_dir}")

🕵️‍♂️ 正在搜尋工作區中的原始 PDF 檔案...
🎯 成功鎖定 4 個原始 PDF 報告。

🚀 啟動『混合式圖表提取引擎』...
📂 正在深度解析報告: World_Inequality_Report_2026.pdf
📂 正在深度解析報告: swissre_sigma-1_2024_english.pdf
📂 正在深度解析報告: Web Version _E-Government Survey 2024 11102024.pdf
📂 正在深度解析報告: natural-catastrophe-and-climate-report-2023.pdf

✨ 第一階段完工！全組共精確捕獲 313 張圖表候選圖片。
📍 實體圖片已擺放在: /home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/extracted_pure_charts

🔮 進入第二階段：啟動官方指定頂級 Kimi 視覺大模型進行數據清洗...
⏩ [1/313] Web_Version__E-Government_Survey_2024_11102024_page_27_img_8.png -> 經 Kimi 鑑定為非圖表物件，已智能過濾跳過。
⏩ [2/313] natural-catastrophe-and-climate-report-2023_page_32_vector.png -> 經 Kimi 鑑定為非圖表物件，已智能過濾跳過。
❌ 處理 World_Inequality_Report_2026_page_169_img_1.jpeg 時發生 API 連線異常: 'NoneType' object has no attribute 'strip'
⏩ [4/313] Web_Version__E-Government_Survey_2024_11102024_page_1_img_11.jpeg -> 經 Kimi 鑑定為非圖表物件，已智能過濾跳過。
⏩ [5/313] World_Inequality_Report_2026_page_167_img_3.png -> 經 Kimi 鑑定為非圖表物件，已智能過濾跳過。
⏩ [6/313] natural-catastrophe-and-climate-report-2023_page_

In [2]:
import os
from openai import OpenAI

# 1. 沿用官方推薦的最強多模態模型與端點
apertus_key = "sk-rc-qLyAj14iCtkrZu9FZuwgYA"
client = OpenAI(
    api_key=apertus_key,
    base_url="https://api.swissai.svc.cscs.ch/v1"
)

caption_dir = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/image_captions_kimi_final"

print("🧠 啟動第二輪優化：針對【無標籤視覺數據 (Unlabeled Data)】進行語義深度精煉...\n")

# 鎖定那 10 個剛剛生成的精準說明書
txt_files = [f for f in os.listdir(caption_dir) if f.endswith('.txt')]

if len(txt_files) == 0:
    print("❌ 錯誤：找不到任何說明的 txt 檔案，請確認路徑是否正確。")
else:
    print(f"📊 偵測到 {len(txt_files)} 個待優化核心圖表文本，開始進行數據區間擴充...")
    optimized_count = 0
    
    for txt_file in txt_files:
        file_path = os.path.join(caption_dir, txt_file)
        with open(file_path, "r", encoding="utf-8") as f:
            current_caption = f.read()
            
        # 斷點續傳：如果已經精煉過就跳過，避免重複消耗 Token
        if "#### [Optimized Range Mapping]" in current_caption:
            print(f"⏩ {txt_file} 之前已經完成過無標籤優化，跳過。")
            continue
            
        print(f"🔄 正在對齊與優化無標籤趨勢: {txt_file}")
        
        # 強迫 Kimi 把「圖表盲區」轉化為 RAG 可搜尋的數字區間短語
        refine_prompt = f"""
        You are an advanced RAG Optimization Engine specializing in handling UNLABELED visual data points from charts.
        Your task is to review the preliminary chart description below and enhance its searchable density for numerical queries.
        
        Preliminary Description:
        \"\"\"
        {current_caption}
        \"\"\"
        
        CRITICAL OPTIMIZATION RULES:
        1. For any estimated, un-labelled, or precise numerical value/percentage mentioned (e.g., 'around 4.5%', '7.1%', 'approximately -0.5%'), automatically append contextually relevant search phrases and numerical ranges.
           - Example: If it says '4.5%', append keywords like '(between 4% and 5%, close to 5%, less than 5%, approximately 4%-5%)'.
           - Example: If a drop goes to 'below 1 billion', append '(less than 1B, under 1,000,000,000, negative growth trend)'.
        2. Create a dedicated section called "#### [Optimized Range Mapping]" at the very bottom of the document to list these expanded, highly searchable range phrases for text embedding lookup.
        
        Return the exact original description text first, followed by this newly generated optimized section.
        """
        
        try:
            response = client.chat.completions.create(
                model="moonshotai/Kimi-K2.5-SDSC",
                messages=[{"role": "user", "content": refine_prompt}],
                max_tokens=800,
                temperature=0.2  # 低隨機性，確保數據推算穩定
            )
            
            optimized_text = response.choices[0].message.content
            
            # 直接覆寫回原檔案，就地升級你們組的圖表文字庫
            with open(file_path, "w", encoding="utf-8") as out_f:
                out_f.write(optimized_text)
            
            optimized_count += 1
            print(f"   ✨ ✅ {txt_file} 數據區間與無標籤防禦優化成功！")
            
        except Exception as e:
            print(f"   ❌ 優化該檔案時發生異常: {e}")

    print(f"\n🎉 完美收工！共成功升級了 {optimized_count} 個圖表的無標籤數據檢索層。")
    print(f"🚀 你們組的核心圖表文字庫現在具備極強的『模糊數字搜尋防禦力』！") 

🧠 啟動第二輪優化：針對【無標籤視覺數據 (Unlabeled Data)】進行語義深度精煉...

📊 偵測到 10 個待優化核心圖表文本，開始進行數據區間擴充...
🔄 正在對齊與優化無標籤趨勢: natural-catastrophe-and-climate-report-2023_page_30_vector_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在對齊與優化無標籤趨勢: World_Inequality_Report_2026_page_190_img_1_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在對齊與優化無標籤趨勢: swissre_sigma-1_2024_english_page_29_vector_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在對齊與優化無標籤趨勢: swissre_sigma-1_2024_english_page_22_vector_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在對齊與優化無標籤趨勢: World_Inequality_Report_2026_page_197_img_1_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在對齊與優化無標籤趨勢: World_Inequality_Report_2026_page_196_img_1_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在對齊與優化無標籤趨勢: World_Inequality_Report_2026_page_173_img_1_caption.txt
   ❌ 優化該檔案時發生異常: write() argument must be str, not None
🔄 正在

In [3]:
import os
import base64
from openai import OpenAI

apertus_key = "sk-rc-qLyAj14iCtkrZu9FZuwgYA"
client = OpenAI(
    api_key=apertus_key,
    base_url="https://api.swissai.svc.cscs.ch/v1"
)

# 定義你的實體圖片與文字說明的目錄路徑
image_dir = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/extracted_pure_charts"
caption_dir = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/image_captions_kimi_final"

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

print("🧠 修正版啟動：帶著實體圖片重新連線 Kimi 視覺引擎，強制進行無標籤數據推算...\n")

txt_files = [f for f in os.listdir(caption_dir) if f.endswith('.txt')]
real_optimized_count = 0

for txt_file in txt_files:
    file_path = os.path.join(caption_dir, txt_file)
    
    # 透過檔名尋找對應的原始圖片名稱
    base_img_name = txt_file.replace("_caption.txt", "")
    possible_extensions = [".png", ".jpg", ".jpeg"]
    img_path = None
    
    for ext in possible_extensions:
        test_path = os.path.join(image_dir, base_img_name + ext)
        if os.path.exists(test_path):
            img_path = test_path
            break
            
    if not img_path:
        print(f"⏩ 找不到 {txt_file} 對應的實體圖片，跳過。")
        continue
        
    with open(file_path, "r", encoding="utf-8") as f:
        current_caption = f.read()
        
    if "#### [Optimized Range Mapping]" in current_caption:
        print(f"⏩ {txt_file} 先前已優化過，跳過。")
        continue
        
    print(f"🔄 正在對齊圖片與文字，深度壓榨無標籤數據: {txt_file}")
    
    # 結合第一輪的初步文字，強迫視覺模型對著圖進行二輪精準數值區間推算
    vlm_refine_prompt = f"""
    You are an expert data retriever. Look at this chart image and your own preliminary analysis below:
    
    Preliminary Analysis:
    {current_caption}
    
    TASK:
    Optimize this description specifically for handling UNLABELED visual data points to maximize RAG search accuracy.
    1. For every estimated, unlabeled, or key numerical point on the bars/lines, provide likely numeric ranges or boundary words.
       - Example: If a point is roughly at 4.5%, explicitly add '(between 4% and 5%, close to 5%, approximately 4-5%)'.
    2. Add a dedicated section at the very end called "#### [Optimized Range Mapping]" listing all searchable numerical range phrases.
    
    Output the original analysis followed by the new optimization section. Do not return empty text.
    """
    
    try:
        base64_image = encode_image(img_path)
        
        # 這次把圖片一起餵進去，對齊 Kimi 的多模態要求
        response = client.chat.completions.create(
            model="moonshotai/Kimi-K2.5-SDSC",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": vlm_refine_prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                    ]
                }
            ],
            max_tokens=800,
            temperature=0.2
        )
        
        result_text = response.choices[0].message.content
        
        # 安全檢查：確保這次真的有吐出字來，才允許寫入
        if result_text and len(result_text.strip()) > 0:
            with open(file_path, "w", encoding="utf-8") as out_f:
                out_f.write(result_text)
            real_optimized_count += 1
            print(f"   ✅ 成功！{txt_file} 已完成無標籤數據區間擴充！")
        else:
            print(f"   ⚠️ 警告：Kimi 這次回傳了空內容。")
            
    except Exception as e:
        print(f"   ❌ 優化失敗原因: {e}")

print(f"\n🏁 真正的流程結束。這次實打實成功升級了 {real_optimized_count} / {len(txt_files)} 個檔案！")

🧠 修正版啟動：帶著實體圖片重新連線 Kimi 視覺引擎，強制進行無標籤數據推算...

🔄 正在對齊圖片與文字，深度壓榨無標籤數據: natural-catastrophe-and-climate-report-2023_page_30_vector_caption.txt
   ⚠️ 警告：Kimi 這次回傳了空內容。
🔄 正在對齊圖片與文字，深度壓榨無標籤數據: World_Inequality_Report_2026_page_190_img_1_caption.txt
   ✅ 成功！World_Inequality_Report_2026_page_190_img_1_caption.txt 已完成無標籤數據區間擴充！
🔄 正在對齊圖片與文字，深度壓榨無標籤數據: swissre_sigma-1_2024_english_page_29_vector_caption.txt
   ⚠️ 警告：Kimi 這次回傳了空內容。
🔄 正在對齊圖片與文字，深度壓榨無標籤數據: swissre_sigma-1_2024_english_page_22_vector_caption.txt
   ✅ 成功！swissre_sigma-1_2024_english_page_22_vector_caption.txt 已完成無標籤數據區間擴充！
🔄 正在對齊圖片與文字，深度壓榨無標籤數據: World_Inequality_Report_2026_page_197_img_1_caption.txt
   ✅ 成功！World_Inequality_Report_2026_page_197_img_1_caption.txt 已完成無標籤數據區間擴充！
🔄 正在對齊圖片與文字，深度壓榨無標籤數據: World_Inequality_Report_2026_page_196_img_1_caption.txt
   ✅ 成功！World_Inequality_Report_2026_page_196_img_1_caption.txt 已完成無標籤數據區間擴充！
🔄 正在對齊圖片與文字，深度壓榨無標籤數據: World_Inequality_Report_2026_page_173_img_1_caption.txt
   ✅ 成功！World_Inequality_Re

In [4]:
import os

# 定義你的乾淨圖表描述路徑
clean_caption_dir = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/image_captions_kimi_final"

print("📂 正在加載已完成『無標籤數值擴充』的頂級圖表文字庫...")

rag_chart_chunks = []
txt_files = [f for f in os.listdir(clean_caption_dir) if f.endswith('.txt')]

for txt_file in txt_files:
    file_path = os.path.join(clean_caption_dir, txt_file)
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # 封裝成標準的 RAG 資料結構，並打上強標籤，告訴後續的 LLM 這是極度精準的圖表數據
    rag_chart_chunks.append({
        "page_content": f"[CRITICAL VISUAL DATA CHART - SOURCE: {txt_file}]\n{content}",
        "metadata": {
            "source": "vlm_chart_caption",
            "original_file": txt_file
        }
    })

print(f"\n✅ 成功！已將 {len(rag_chart_chunks)} 個多模態圖表高密度區塊封裝完畢！")
print("💡 指引：現在你可以直接用 `rag_chart_chunks` 去 extend（合併）你們組原本切好的 PDF 純文字 Chunks 列表了。")
print("🚀 當主辦單位的測試集問到數字時，這些 Chunks 會帶著那些 (between 4% and 5%) 的關鍵字，以極高的相似度得分被當場檢索出來！")

📂 正在加載已完成『無標籤數值擴充』的頂級圖表文字庫...

✅ 成功！已將 10 個多模態圖表高密度區塊封裝完畢！
💡 指引：現在你可以直接用 `rag_chart_chunks` 去 extend（合併）你們組原本切好的 PDF 純文字 Chunks 列表了。
🚀 當主辦單位的測試集問到數字時，這些 Chunks 會帶著那些 (between 4% and 5%) 的關鍵字，以極高的相似度得分被當場檢索出來！


In [ ]:
BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch/"
leaderboard_endpoint = f"{BASE_URL}/api/v1/leaderboard"
submit_endpoint = f"{BASE_URL}/api/v1/submit"


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(leaderboard_endpoint).json()
    return pd.DataFrame(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str) -> pd.Series:
    if user == "your_team_name" or user == "":
        raise ValueError("Please set your team name in the 'TEAM_NAME' variable.")
    predictions = df[["question", "answer"]].to_dict(orient="records")
    submission = {"predictions": predictions}
    response = requests.post(submit_endpoint, json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        response.raise_for_status()
    print(response.json())
    return pd.Series(response.json())